# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.

- 이번 과제는 LMS 안내에 따라 공통 `data/raw/` 대신 `practice/chapter05/data/raw/`를 사용함
- 제출 폴더에서 실행할 때는 인접한 `llm-data-analysis-course`의 전용 Raw를 읽고 제출 폴더 `reproduction/`의 모듈과 스크립트를 사용함


In [1]:
from pathlib import Path
import sys
import os
import subprocess
import hashlib
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()
candidate_roots = [CURRENT_DIR, *CURRENT_DIR.parents]
candidate_roots.extend(parent / "llm-data-analysis-course" for parent in CURRENT_DIR.parents)
PROJECT_ROOT = next((path for path in candidate_roots if (path / "practice/chapter05/data/raw").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Chapter 05 전용 Raw 경로를 찾지 못했습니다.")
RAW_DIR = PROJECT_ROOT / "practice/chapter05/data/raw"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
REPORT_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
support_candidates = [p / "reproduction" for p in [CURRENT_DIR, *CURRENT_DIR.parents]]
support_candidates += [p / "chapter05/reproduction" for p in [CURRENT_DIR, *CURRENT_DIR.parents]]
SUPPORT_DIR = next((p for p in support_candidates if (p / "scripts/preprocess_data.py").is_file()), None)
if SUPPORT_DIR is None:
    raise FileNotFoundError("제출 폴더의 chapter05/reproduction 코드가 필요합니다.")
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("현재 실행 위치(프로젝트 기준):", Path(os.path.relpath(CURRENT_DIR, PROJECT_ROOT)))
print("프로젝트 루트: .")
print("원본 데이터 폴더:", RAW_DIR.relative_to(PROJECT_ROOT))
print("전처리 데이터 폴더:", PROCESSED_DIR.relative_to(PROJECT_ROOT))
print("보고서 폴더:", REPORT_DIR.relative_to(PROJECT_ROOT))


Python: 3.12.14
pandas: 3.0.5
numpy: 2.5.3
현재 실행 위치(프로젝트 기준): ../llm-data-analysis-study/chapter05
프로젝트 루트: .
원본 데이터 폴더: practice/chapter05/data/raw
전처리 데이터 폴더: data/processed
보고서 폴더: reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')

raw_hashes_before = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(RAW_DIR.glob("*.csv"))
}
print("원본 파일 SHA-256 기록:", raw_hashes_before)


원본 데이터 불러오기 완료
원본 파일 SHA-256 기록: {'customers.csv': 'ef4e554eaae1bb0cb7605aa9e4503aad1e68bbf1eae9093def5c2d04bd26b8cc', 'order_items.csv': 'e59b236328b6b094fdcbc0142af35b7ccb54e4246e1a57180c39997582303e5f', 'orders.csv': 'bb32cb334b9e4920f68e6ef55df17d414bdc51f7c28a58fa3bf1bfb6b2aa5fa1', 'products.csv': 'cad0a825a29078d9451aa74fdfa9e61d474f442d2e377839eca24e9595debbf0'}


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
customer_id,0,0.00
name,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [8]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include=['object', 'str']).columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
products_clean["category"] = products_clean["category"].str.replace(r"\s+", "", regex=True)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')
blank_string_counts = {
    name: {
        column: int(df[column].astype("string").str.strip().eq("").sum())
        for column in df.select_dtypes(include=["object", "str"]).columns
    }
    for name, df in raw_data.items()
}
print("원본 공백 문자열 개수:", blank_string_counts)
print("문자열 정리 후 결측 셀 수:", {
    name: int(df.isna().sum().sum())
    for name, df in {"customers": customers_clean, "products": products_clean,
                     "orders": orders_clean, "order_items": order_items_clean}.items()
})


문자열 앞뒤 공백 제거 완료
원본 공백 문자열 개수: {'customers': {'name': 0, 'gender': 0, 'age': 0, 'city': 0, 'signup_date': 0}, 'products': {'product_name': 0, 'category': 0, 'price': 0}, 'orders': {'order_date': 0, 'payment_method': 0, 'order_status': 0}, 'order_items': {'quantity': 0, 'unit_price': 0}}
문자열 정리 후 결측 셀 수: {'customers': 0, 'products': 0, 'orders': 0, 'order_items': 0}


In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

### 주문 상태와 상품 범주 허용값 확인

In [15]:
allowed_status = {"completed", "cancelled", "refunded"}
allowed_categories = {"도서", "뷰티", "생활용품", "스포츠", "식품", "전자기기", "패션"}
status_outside = sorted(set(orders_clean["order_status"].dropna()) - allowed_status)
category_outside = sorted(set(products_clean["category"].dropna()) - allowed_categories)
print("표준 상태값 밖:", status_outside)
print("상품 범주 허용값 밖:", category_outside)
print("상품 범주 종류:", products["category"].nunique(), "→", products_clean["category"].nunique())
display(products_clean["category"].value_counts())

표준 상태값 밖: []
상품 범주 허용값 밖: []
상품 범주 종류: 13 → 7


category
뷰티      18
패션      17
식품      16
도서      14
전자기기    14
스포츠     11
생활용품    10
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [16]:
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
print("order_date 변환 실패(원본 비결측 중 변환 불가):", int(orders_clean["order_date"].isna().sum() - orders["order_date"].isna().sum()))
print("주문 시작일:", orders_clean["order_date"].min())
print("주문 종료일:", orders_clean["order_date"].max())
orders_clean["order_month"] = orders_clean["order_date"].dt.to_period("M").astype("string")
orders_clean["order_dayofweek"] = orders_clean["order_date"].dt.day_name()
orders_clean[["order_date", "order_month", "order_dayofweek"]].head()


order_date 변환 실패(원본 비결측 중 변환 불가): 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [17]:
if "signup_date" in customers_clean.columns:
    customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"], errors="coerce")
    print("signup_date 변환 실패(원본 비결측 중 변환 불가):", int(customers_clean["signup_date"].isna().sum() - customers["signup_date"].isna().sum()))
    print("가입 시작일:", customers_clean["signup_date"].min())
    print("가입 종료일:", customers_clean["signup_date"].max())
    display(customers_clean[["customer_id", "signup_date"]].head())


signup_date 변환 실패(원본 비결측 중 변환 불가): 2
가입 시작일: 2023-07-21 00:00:00
가입 종료일: 2026-08-24 00:00:00


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [18]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

numeric_sources = {
    "age": (customers["age"], "고객"),
    "price": (products["price"], "상품"),
    "quantity": (order_items["quantity"], "주문상세"),
    "unit_price": (order_items["unit_price"], "주문상세"),
}
for column, (raw_series, _) in numeric_sources.items():
    converted = to_number(raw_series)
    print(f"{column} 변환 실패(원본 비결측 중 변환 불가):", int(converted.isna().sum() - raw_series.isna().sum()))

products_clean["price"] = to_number(products_clean["price"])
order_items_clean["quantity"] = to_number(order_items_clean["quantity"])
order_items_clean["unit_price"] = to_number(order_items_clean["unit_price"])


age 변환 실패(원본 비결측 중 변환 불가): 2
price 변환 실패(원본 비결측 중 변환 불가): 2
quantity 변환 실패(원본 비결측 중 변환 불가): 5
unit_price 변환 실패(원본 비결측 중 변환 불가): 2


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [19]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [20]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [21]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [22]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [23]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [24]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [25]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


### 전처리 전후 품질 지표

In [26]:
quality_comparison = pd.DataFrame({
    "dataset": list(raw_data),
    "missing_raw": [int(df.isna().sum().sum()) for df in raw_data.values()],
    "missing_processed": [int(processed_data[name].isna().sum().sum()) for name in raw_data],
    "duplicate_raw": [int(df.duplicated().sum()) for df in raw_data.values()],
    "duplicate_processed": [int(processed_data[name].duplicated().sum()) for name in raw_data],
})
display(quality_comparison)
print("전체 결측:", quality_comparison["missing_raw"].sum(), "→", quality_comparison["missing_processed"].sum())
print("전체 완전 중복:", quality_comparison["duplicate_raw"].sum(), "→", quality_comparison["duplicate_processed"].sum())

,dataset,missing_raw,missing_processed,duplicate_raw,duplicate_processed
0,customers,6,2,2,0
1,products,0,0,2,0
2,orders,0,9,2,0
3,order_items,0,0,2,0


전체 결측: 6 → 11
전체 완전 중복: 8 → 0


### 기본 키와 관계 오류 원인 확인

In [27]:
key_columns = {"customers": "customer_id", "products": "product_id",
               "orders": "order_id", "order_items": "order_item_id"}
key_rows = []
for name, key in key_columns.items():
    raw, clean = raw_data[name], processed_data[name]
    key_rows.append({"dataset": name, "raw_key_missing": int(raw[key].isna().sum()),
                     "raw_key_duplicate": int(raw[key].duplicated().sum()),
                     "key_duplicate_after_full_dedup": int(raw.drop_duplicates()[key].duplicated().sum()),
                     "clean_key_missing": int(clean[key].isna().sum()),
                     "clean_key_duplicate": int(clean[key].duplicated().sum())})
display(pd.DataFrame(key_rows))
relations = [("orders", "customer_id", "customers", "customer_id"),
             ("order_items", "order_id", "orders", "order_id"),
             ("order_items", "product_id", "products", "product_id")]
display(pd.DataFrame([
    {"relation": f"{child}.{fk} → {parent}.{pk}",
     "raw_invalid": int((~raw_data[child][fk].isin(raw_data[parent][pk])).sum()),
     "clean_invalid": int((~processed_data[child][fk].isin(processed_data[parent][pk])).sum())}
    for child, fk, parent, pk in relations
]))
removed_product_ids = sorted(set(products["product_id"]) - set(products_clean["product_id"]))
print("제외된 상품 ID:", removed_product_ids)
print("제외된 상품을 참조하는 처리 후 상세 행:", int(order_items_clean["product_id"].isin(removed_product_ids).sum()))
print("원본에도 없는 상품을 참조하는 처리 후 상세 행:", int((~order_items_clean["product_id"].isin(products["product_id"])).sum()))
assert order_items_clean["line_total"].eq(order_items_clean["quantity"] * order_items_clean["unit_price"]).all()
print("전체 상세 행의 line_total 계산식 일치: PASS")

,dataset,raw_key_missing,raw_key_duplicate,key_duplicate_after_full_dedup,clean_key_missing,clean_key_duplicate
0,customers,0,2,0,0,0
1,products,0,2,0,0,0
2,orders,0,2,0,0,0
3,order_items,0,2,0,0,0


,relation,raw_invalid,clean_invalid
0,orders.customer_id → customers.customer_id,2,2
1,order_items.order_id → orders.order_id,2,2
2,order_items.product_id → products.product_id,2,26


제외된 상품 ID: [18, 48, 73, 95]
제외된 상품을 참조하는 처리 후 상세 행: 24
원본에도 없는 상품을 참조하는 처리 후 상세 행: 2
전체 상세 행의 line_total 계산식 일치: PASS


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [28]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

[path.relative_to(PROJECT_ROOT) for path in sorted(PROCESSED_DIR.glob('*_clean.csv'))]

[PosixPath('data/processed/customers_clean.csv'),
 PosixPath('data/processed/order_items_clean.csv'),
 PosixPath('data/processed/orders_clean.csv'),
 PosixPath('data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [29]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path.relative_to(PROJECT_ROOT))


요약 보고서 저장 완료: reports/ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [30]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)
    result['category'] = result['category'].str.replace(r'\s+', '', regex=True)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype('string')
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [31]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)
function_data = {
    "customers": customers_clean_func, "products": products_clean_func,
    "orders": orders_clean_func, "order_items": order_items_clean_func,
}
for name in processed_data:
    pd.testing.assert_frame_equal(
        processed_data[name].reset_index(drop=True),
        function_data[name].reset_index(drop=True),
    )
print("단계별 처리와 함수 결과 전체 일치: PASS")


(150, 6)
(96, 4)
(300, 7)
(752, 6)


단계별 처리와 함수 결과 전체 일치: PASS


## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [32]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [33]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [34]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.

- 제출용 수정 모듈·스크립트와 단독 실행 방법은 [재현용 코드 안내](reproduction/README.md)에 포함함
- 아래 셀은 같은 전용 Raw와 출력 경로를 명시해 제출용 스크립트를 실행함


In [35]:
run_result = subprocess.run(
    [sys.executable, str(SUPPORT_DIR / "scripts/preprocess_data.py"),
     "--raw-dir", str(RAW_DIR), "--output-dir", str(PROCESSED_DIR),
     "--report-dir", str(REPORT_DIR)],
    cwd=SUPPORT_DIR, capture_output=True, text=True, check=False
)
print(run_result.stdout)
if run_result.stderr:
    print(run_result.stderr)
if run_result.returncode != 0:
    raise RuntimeError(f"전처리 스크립트 실패: {run_result.returncode}")
print("스크립트 종료 코드:", run_result.returncode)


전처리 완료
입력: --raw-dir로 지정한 Chapter 05 전용 Raw CSV 4개

[전처리 전후 데이터 크기]
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       152            6             150                  6
order_items       765            5             752                  6
     orders       302            5             300                  7
   products       102            4              96                  4

[중복 점검 결과]
    dataset  row_duplicate_count    key_column  key_duplicate_count
  customers                    0   customer_id                    0
   products                    0    product_id                    0
     orders                    0      order_id                    0
order_items                    0 order_item_id                    0

[파일 간 관계 점검 결과]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              2
      order_items.order_id exists in orders.order_id              2
order_ite

In [36]:
saved_data = {name: pd.read_csv(PROCESSED_DIR / f"{name}_clean.csv") for name in processed_data}
for name, frame in saved_data.items():
    path = PROCESSED_DIR / f"{name}_clean.csv"
    print(name, frame.shape, "columns:", frame.columns.tolist(), "size:", path.stat().st_size)
saved_orders = saved_data["orders"]
saved_orders["order_date"] = pd.to_datetime(saved_orders["order_date"], errors="coerce")
saved_data["customers"]["signup_date"] = pd.to_datetime(saved_data["customers"]["signup_date"], errors="coerce")
print("저장된 order_date 결측(재변환 후):", int(saved_orders["order_date"].isna().sum()))
print("CSV 저장 전 order_date 결측:", int(orders_clean["order_date"].isna().sum()))
for name, expected in processed_data.items():
    pd.testing.assert_frame_equal(
        expected.reset_index(drop=True),
        saved_data[name].reset_index(drop=True),
        check_dtype=False,
    )
print("Notebook 처리 결과와 스크립트 CSV 값 전체 일치: PASS")
raw_hashes_after = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(RAW_DIR.glob("*.csv"))
}
print("raw 원본 SHA-256 동일:", raw_hashes_before == raw_hashes_after)
assert raw_hashes_before == raw_hashes_after


customers (150, 6) columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date'] size: 5773
products (96, 4) columns: ['product_id', 'product_name', 'category', 'price'] size: 3871
orders (300, 7) columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek'] size: 16120
order_items (752, 6) columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total'] size: 23918
저장된 order_date 결측(재변환 후): 3
CSV 저장 전 order_date 결측: 3
Notebook 처리 결과와 스크립트 CSV 값 전체 일치: PASS
raw 원본 SHA-256 동일: True


### 저장된 전처리 보고서 확인

In [37]:
final_report = (REPORT_DIR / "ch05_preprocessing_summary.md").read_text(encoding="utf-8")
assert final_report.strip()
print(final_report)
print("요약 보고서 내용 확인: PASS")

# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       152            6             150                  6
order_items       765            5             752                  6
     orders       302            5             300                  7
   products       102            4              96                  4
```

## 중복 점검 결과

```text
    dataset  row_duplicate_count    key_column  key_duplicate_count
  customers                    0   customer_id                    0
   products                    0    product_id                    0
     orders                    0      order_id                    0
order_items                    0 order_item_id                    0
```

## 파일 간 관계 점검 결과

```text
                                               check  invalid_count
  orders.customer_id exists in cus

## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [38]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합칩니다.
missing_tables = []
for name, df in raw_data.items():
    part = missing_summary(df).reset_index(names="column")
    part.insert(0, "dataset", name)
    missing_tables.append(part)
all_missing_summary = pd.concat(missing_tables, ignore_index=True)
all_missing_summary


,dataset,column,missing_count,missing_ratio
0,customers,age,4,2.63
1,customers,city,2,1.32
2,customers,customer_id,0,0.00
3,customers,name,0,0.00
4,customers,gender,0,0.00
5,customers,signup_date,0,0.00
6,products,product_id,0,0.00
7,products,product_name,0,0.00
8,products,category,0,0.00
9,products,price,0,0.00


In [39]:
# 과제 2. 나이 18세 미만 또는 100세 초과 값 확인
customers_clean.loc[(customers_clean["age"] < 18) | (customers_clean["age"] > 100), ["customer_id", "age"]].sort_values("age")


,customer_id,age
32,33,150.0


In [40]:
# 과제 3. 전체 상태를 포함한 월별 주문 행 수 확인
orders_clean.groupby("order_month", dropna=False).size().rename("order_count").sort_index()


order_month
2025-07    19
2025-08    26
2025-09    12
2025-10    22
2025-11    23
2025-12    23
2026-01    26
2026-02    19
2026-03    22
2026-04    26
2026-05    20
2026-06    24
2026-07    12
2026-08    21
2027-01     1
2028-12     1
<NA>        3
Name: order_count, dtype: int64

In [41]:
# 과제 4. 주문 상세 line_total 상위 10건 확인
order_items_clean.nlargest(10, "line_total")[["order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_total"]]


,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
9,10,5,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
116,117,51,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
10,11,6,97,5.0,187000.0,935000.0


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

> 주 제출물은 실행 완료 Notebook `chapter05/chapter05.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
이름: 장태원  
GitHub ID: pmsky1  
작성일: 2026-10-11  
최종 제출 URL: https://github.com/pmsky1/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb  
사용한 AI 도구: ChatGPT

## 1. 전처리 전 상태 기록
- 각 데이터 shape: 고객 152×6, 상품 102×4, 주문 302×5, 주문상세 765×5
- 결측치: 고객 age 4건(2.63%), city 2건(1.32%), 다른 데이터는 0건
- 전체 중복: 네 데이터셋에서 각각 2행씩, 총 8행
- 주요 ID 중복: 각 기본 ID에서 2건씩 확인했고 모두 완전 중복 행과 같은 행이었음
- 타입 문제 후보: age·price·quantity·unit_price와 날짜가 문자열로 읽혔고, 상태값과 상품 범주 표기가 섞여 있었음

### Evidence
- 위 원본 크기·결측치·중복·기본 ID 중복 코드 셀의 저장된 Output에서 각 데이터 행 수와 고객 결측 6셀, 테이블별 중복 2행을 확인함

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
- 고객 데이터의 결측 6건은 age 4건과 city 2건이며, 원본 문자열 컬럼에는 공백만 있는 값은 없었음
- 숫자 변환이 필요한 값은 age 2건, price 2건, quantity 5건, unit_price 2건이 숫자로 바뀌지 않았고, 날짜는 signup_date 2건과 order_date 3건이 변환되지 않았음
- 세 테이블의 참조 관계 모두 원본에서 연결되지 않는 행이 각각 2건 있었음

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지와 이유를 작성하세요.
- 먼저 원본을 보존하고 완전 중복과 기본 ID 중복을 확인한 뒤 타입과 참조 관계를 처리하는 것이 안전하다고 판단함
- 기본 ID 중복은 모두 동일한 전체 행이 반복된 경우라 완전 중복 행 제거로 해결할 수 있었지만, 연결되지 않는 외래키는 원인을 알 수 없어 삭제하지 않고 남김

### 업무·분석적 의미
- 중복 행이 남으면 고객·상품·주문 수와 금액이 부풀 수 있고, 타입 오류가 남으면 계산이나 기간 집계가 누락될 수 있음
- 고객 정보 누락과 연결되지 않는 키는 고객 분석과 주문 상세 집계의 신뢰도에 직접 영향을 줌

### 한계와 추가 확인 사항
- 결측이나 잘못된 형식이 생긴 원인은 CSV만으로 알 수 없어 수집 시스템이나 담당자 확인이 필요함
- 나이 150세와 2027·2028년 주문일처럼 범위를 벗어나 보이는 값도 업무상 오류인지 현재 자료만으로 단정하지 않음

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 앞뒤 공백을 제거하고 상품 category의 내부 공백 표기를 도서·뷰티·생활용품·스포츠·식품·전자기기·패션 7개 범주로 통일함
- 숫자 변환 대상과 실패 건수: age 2건, price 2건, quantity 5건, unit_price 2건
- 날짜 변환 대상과 실패 건수: signup_date 2건, order_date 3건
- 날짜 범위: 가입일 2023-07-21~2026-08-24, 주문일 2025-07-01~2028-12-31임
- 결측 처리 기준: age의 기존 결측 4건과 변환 실패 2건은 중앙값 41로 대체하고 city 결측 2건은 Unknown으로 표시함

### Evidence
- 위 문자열 정리·날짜·숫자 변환 코드 셀의 저장된 Output에서 공백 문자열 0건, 날짜 실패 2·3건과 숫자 실패 2·2·5·2건을 확인함

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
- 원본의 공백 표기와 주문 상태의 대소문자·한글 표기를 정리한 뒤 주문 상태는 completed 180건, cancelled 63건, refunded 57건으로 통일됨
- 숫자 변환 실패 및 0 이하 가격·수량·단가 기준에 따라 상품은 102행에서 96행, 주문상세는 765행에서 752행으로 줄었음
- 변환할 수 없는 signup_date 2건과 order_date 3건은 NaT로 남겼고, 임의 날짜로 대체하지 않음

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.
- age 중앙값 대체는 고객 행을 보존하지만 실제 나이 분포를 매끈하게 바꿀 수 있어 대체 대상 6건을 요약에 기록함
- 도시 Unknown은 미상이라는 정보를 유지하기 위한 구분값이며 실제 도시로 추정하지 않음
- 가격·수량 변환 실패와 0 이하 값은 정상 주문 분석에서 제외했지만, 무료 상품이나 반품일 가능성을 배제할 수 없어 실습 기준으로만 적용함

### 업무·분석적 의미
- 표기 차이를 정리하면 같은 상태나 상품 범주가 여러 집단으로 나뉘어 집계되는 일을 줄일 수 있음
- 날짜 변환 실패 5건은 기간 분석에서 제외될 수 있으므로 결과를 해석할 때 누락 규모를 함께 봐야 함

### 한계와 추가 확인 사항
- 제외한 상품과 주문상세가 증정·반품·수기 입력 오류 중 무엇인지는 원본 시스템에서 확인해야 함
- 나이 중앙값 대체 결과와 age 150 값은 고객 연령 분석 전에 민감도와 업무 기준을 다시 확인해야 함

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 각 테이블의 완전 중복 2행씩 제거했고 기본 ID 중복은 제거 후 0건임
- 표준화한 범주값: order_status를 completed·cancelled·refunded로 통일하고 상품 category의 띄어쓰기 변형을 7개 범주로 통일함
- 허용값 밖 값: 표준화 후 허용된 주문 상태 3개와 상품 범주 7개 밖의 값은 각각 0건임
- 이상값 후보: price 0 이하 2건, quantity 0 이하 3건, unit_price 0 이하 1건, age 150 1건, 2027·2028년 주문일 2건

### Evidence
- 위 전체 중복·상태값 표준화·이상값 후보 코드 셀과 아래 실습 과제 출력에서 중복 제거, 상태 3종, 0 이하 값과 age 150을 확인함

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
- order_items.order_id 반복은 기본 ID 중복과 달리 한 주문에 여러 상세 행이 연결된 값임
- 주문일 변환 실패 3건은 월별 집계에서 결측으로 남았고, 미래 연도 주문 2건은 원본에 그대로 존재함
- 상품 price 최대 9,999,999와 주문상세 quantity 50·100도 확인했으며 양수 조건을 통과해 유지됨
- quantity 100인 상세 금액은 15,500,000으로, 단위 오류나 실제 대량 주문인지 확인이 필요함

### 나의 해석과 판단
이상값을 자동 삭제하지 않고 후보로 검토해야 하는 이유를 작성하세요.
- 완전히 같은 행은 중복으로 제거했지만 order_items를 order_id 기준으로 중복 제거하면 여러 상품 상세가 사라지므로 적용하지 않음
- 이상값을 오류로 확정하지 않았으며, 0 이하 가격·수량·단가는 실습 기준으로 제외하고 age 150과 큰 양수 값은 유지함
- 음수 수량은 반품, 0원 가격은 증정일 수 있어 제외 기준의 업무 확인이 필요함

### 업무·분석적 의미
- 범주 표기 통일은 주문 상태별·상품 범주별 집계의 일관성을 높임
- 자동 삭제 기준을 넓게 적용하면 반품·증정·미래 예약 주문 같은 유효한 업무 기록을 잃을 수 있음

### 한계와 추가 확인 사항
- 주문 상태별 취소·환불 처리 규칙과 미래 주문일의 의미를 데이터 제공처에 확인해야 함

## 4. PK/FK와 파생 컬럼 재검증
- 없는 customer_id: 2건
- 없는 order_id: 2건
- 없는 product_id: 26건
- 생성한 파생 컬럼: order_month, order_dayofweek, line_total
- line_total 검증: 양수 quantity와 unit_price의 곱으로 계산했고 합계는 252,140,000임

### Evidence
- 위 line_total 계산·합계·관계 검사 코드 셀의 저장된 Output에서 계산 예시와 합계, 외래키 미연결 2·2·26건을 확인함

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
- 처리 후 orders.customer_id 연결 실패 2건, order_items.order_id 연결 실패 2건, order_items.product_id 연결 실패 26건을 확인함
- 상품 ID 18·48·73·95를 참조하는 주문상세 24건은 가격 변환 실패 또는 0 이하 가격으로 상품 행이 제외된 뒤 연결이 끊겼고, 나머지 2건은 원본에 없는 상품 ID였음

### 나의 해석과 판단
전처리 후 관계를 다시 검증해야 하는 이유를 작성하세요.
- 관계 검증 결과를 0으로 만들기 위해 상세 행을 임의 삭제하지 않고 실패 건수와 원인을 구분해 기록함
- line_total 합계는 모든 주문 상태를 포함하고 할인·배송비·환불 조정도 반영하지 않으므로 매출액이나 순매출로 부르지 않음

## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 고객 152, 상품 102, 주문 302, 상세 765 | 고객 150, 상품 96, 주문 300, 상세 752 | 완전 중복 제거, 유효하지 않은 가격·수량·단가 행 제외 |
| 결측 | 총 6셀 | 총 11셀 | 날짜 변환 실패 5건을 보존했고 주문 월·요일 파생 컬럼에도 결측이 생김 |
| 중복 | 각 테이블 2행, 총 8행 | 0행 | 완전히 동일한 행 제거 |
| 변환 실패 | 숫자 11건, 날짜 5건 | 숫자 오류는 대체 또는 행 제외, 날짜 5건은 NaT로 유지 | age 오류는 중앙값 대체, 비정상 가격·수량·단가는 실습 기준 적용 |

### Evidence
- 위 전처리 전후 크기 표와 품질 지표 표의 저장된 Output에서 네 데이터의 행 수, 전체 결측 6→11셀, 전체 중복 8→0행을 확인함

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
- 고객은 152→150행, 상품은 102→96행, 주문은 302→300행, 주문상세는 765→752행으로 변경됨
- 처리 후 결측 11셀은 signup_date 2건과 order_date 3건, 파생 월 3건, 파생 요일 3건으로 구성됨

### 나의 해석과 판단
행 수나 값 분포 변화가 의도한 결과인지 판단한 근거를 작성하세요.
- 행 수 감소는 중복 및 실습 기준상 제외한 숫자 오류·비정상 값으로 설명할 수 있음
- 결측 셀 수가 6에서 11로 늘어난 것은 날짜 변환 실패를 감추지 않고 유지했으며 파생 컬럼에도 같은 결측이 반영됐기 때문임

### 업무·분석적 의미
- 처리 전후 행 수만 비교하지 않고 결측·중복·변환 실패 이유를 함께 남겨야 데이터 손실을 설명할 수 있음

### 한계와 추가 확인 사항
- 가격·수량 제외 기준을 실제 업무에 적용하기 전에는 반품·증정·예약 거래를 포함할지 확인해야 함

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: data/processed/customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md
- 재현에 사용한 코드: 제출 폴더의 reproduction/src/와 reproduction/scripts/preprocess_data.py이며 전용 Raw·출력 경로를 인자로 지정함
- python scripts/preprocess_data.py 재실행 결과: 종료 코드 0, 네 CSV와 보고서 생성
- 재실행 후 달라진 점: Notebook 전처리 결과와 스크립트 저장 CSV의 전체 값 비교가 모두 일치했고 raw CSV 4개의 SHA-256도 실행 전후 동일함

### Evidence
- 위 스크립트 실행 코드 셀에 종료 코드 0과 CSV·보고서 경로가 남았고, 저장 결과 재확인 코드 셀에 전체 값 일치 PASS와 raw 해시 동일 True가 남음

![스크립트 실행 결과](images/step06_script.png)

![스크립트 재현성 검증](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것의 차이를 작성하세요.
- 한 번 실행한 Notebook은 당시 결과를 보여주고, 스크립트 재실행 및 저장 파일 재검증은 같은 규칙으로 결과를 다시 만들 수 있는지 확인함
- 재실행으로 관계 오류가 사라진 것은 아니며 고객 2건·주문 2건·상품 26건의 연결 문제는 보고서에도 남음

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. raw를 보존하고 처리 전 상태와 파일 해시를 기록함
2. 값의 의미를 모르면 임의 대체·삭제하지 않고 변환 실패와 관계 오류를 남김
3. 처리 뒤 행 수·결측·중복·PK/FK와 저장 결과를 다시 확인함

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
- 0 이하 quantity를 일괄 삭제하는 처리임
- 반품이나 취소 후 회수 같은 거래가 음수·0으로 표현될 수 있어 업무 확인 없이 삭제하면 실제 기록을 잃을 수 있음

### 다음 EDA에서 특히 주의할 데이터 특성
- 주문 상태에 completed·cancelled·refunded가 모두 있으며 월별 주문 수는 전체 상태를 포함함
- 날짜 변환 실패 3건과 2027·2028년 날짜가 있고 고객·주문·상품 키 연결 오류도 남아 있음
- line_total 합계를 매출로 해석하지 않고 상태와 연결 성공 여부를 먼저 구분해야 함

### 현재 전처리의 한계
- 변환되지 않은 날짜 5건과 세 관계 검사에서 합계 30건으로 집계된 연결 실패의 실제 업무 의미를 원본 시스템에서 확인하지 못함
- age 150을 포함한 범위 밖 값과 가격·수량 제외 기준을 도메인 담당자에게 검증하지 못함

### LLM 검토 Prompt와 직접 검증

- 다음 집계 정보만 사용하고 실제 고객 행이나 이름은 넣지 않음

> 쇼핑몰 전처리 기준을 검토해 주세요. 실제 고객 행이나 이름은 제공하지 않고 요약 통계만 사용합니다  
> customers 152행, age 결측 4건, city 결측 2건, age 숫자 변환 실패 2건, age 150 한 건  
> products 102행, price 변환 실패 2건, price 0 이하 2건, category 띄어쓰기 변형  
> orders 302행, order_date 변환 실패 3건, 상태 표기 변형, customer 외래키 오류 2건  
> order_items 765행, quantity 변환 실패 5건 및 0 이하 3건, unit_price 변환 실패 2건 및 0 이하 1건  
> 처리 후 customer 외래키 오류 2건, order 외래키 오류 2건, product 외래키 오류 26건  
> 결측치·이상값·중복·외래키를 어떻게 처리할지 후보와 정보 손실 가능성을 제안하고, 사람이 결과를 검증할 항목을 구분해 주세요. 업무 의미를 모르는 값은 오류라고 단정하지 말고 raw를 덮어쓰지 않는 기준으로 답해 주세요

### LLM 제안에 대한 검증
- 나이 중앙값 대체는 고객 행을 보존하지만 연령 분포를 바꾸므로 대체 여부를 기록하고 age 150을 별도 검토하라는 제안을 실제 데이터와 대조함
- 0 이하 가격·수량과 변환 실패는 상품·반품 업무 의미를 확인한 뒤 처리해야 한다는 제안을 반영해 연결 오류를 자동 삭제하지 않고 남김
- 날짜 변환 실패와 관계 오류가 실제 출력에 남는지 확인했고 raw 파일 해시, CSV 전체 값 비교, 스크립트 종료 코드를 직접 검증함

### 최종 LLM 코드 검토 반영

- 제출 폴더에서 경로를 표시할 때 발생할 수 있는 오류를 수정하고 해당 위치에서 전체 셀을 실행해 확인함
- 함수 버전에도 상품 범주 공백 통일과 날짜 결측 유지 기준을 적용하고 단계별 처리 결과와 전체 값을 비교함
- 저장 CSV의 날짜 결측 3건은 기존 변환 실패가 유지된 값이며 저장 과정에서 새로 발생한 실패와 구분함

## Chapter 05 완료 Evidence
- Notebook 실행: PASS, 코드 셀 오류 출력 0건
- raw 원본 보존 확인: PASS, CSV 4개 SHA-256 실행 전후 동일
- processed CSV 4개 생성: PASS
- 숫자/날짜 변환 실패 점검: PASS, 숫자 11건과 날짜 5건 확인
- 전체 중복/키 중복 점검: PASS, 처리 후 각 테이블 0건
- PK/FK 관계 검증: PASS, 검사 결과 미연결 고객 2건·주문 2건·상품 26건은 미해결
- 파생 컬럼 확인: PASS, order_month·order_dayofweek·line_total 확인
- preprocess_data.py 재실행: PASS, 종료 코드 0
- ch05_preprocessing_summary.md 생성: PASS
- 저장 결과 재확인: PASS, Notebook 결과와 CSV 전체 값 일치

### 내가 적용한 전처리 기준
- 완전 중복은 제거하고 나이 결측·변환 실패는 중앙값 41로 대체했으며 비정상 가격·수량·단가는 실습 기준으로 제외함

### 전처리 전후 달라진 점
- 고객 152→150행, 상품 102→96행, 주문 302→300행, 주문상세 765→752행
- 전체 결측은 6→11셀, 완전 중복은 8→0행이며 날짜 실패 5건은 결측으로 남음

### 아직 업무 확인이 필요한 항목
- 세 관계의 미연결 2·2·26건, age 150, 미래 주문일 2건, 큰 가격·수량과 0 이하 가격·수량·단가의 실제 의미

### 제출 전 추가 LLM 검토

- Prompt 요약: 전처리 전 상태, 날짜·숫자 실패, 중앙값 대체와 행 제외 기준, 처리 후 크기·결측·미연결 결과를 제공하고 정보 손실 위험과 추가 검증 항목을 요청함
- ChatGPT 응답: raw 보존과 완전 중복 제거는 타당하며, 가격·수량 행 제외가 분포를 바꾸고 상품 미연결이 26건으로 증가한 점을 주의해야 한다고 제안함
- 직접 검증과 판단: 상품 제외로 연결이 끊긴 상세 24건과 원래 없는 상품 참조 2건을 구분했고, age 150과 미래 주문일은 유지함. 미연결을 없애기 위한 추가 삭제는 보류하고 후속 분석 전에 제외 기준과 결과 영향을 확인할 필요가 있다고 판단함

![추가 LLM Prompt와 응답](images/step07_llm_review.png)

## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [ ] chapter05/chapter05.ipynb가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.